# Prove MinIO Deletion Works Without STS

> **New here? Read this first.** This notebook proves *why* Polaris `purge` leaves
> orphaned files in MinIO in our no-STS environment — and that the files can still
> be deleted correctly using the static MinIO credentials directly.

## Background (the problem in one paragraph)

When you drop an Iceberg table with `?purgeRequested=true`, Polaris returns `204`
and removes the catalog record, but the data/metadata files are **not deleted** from
MinIO — they orphan permanently. Root cause: Polaris schedules an async cleanup task
that needs *write-scoped* storage credentials. Our deployment has **no STS**, so the
task receives an empty write-location set (`allowedWriteLocations=[]`), fails, and
retries forever. This matches Apache Polaris **issue #379** (the cleanup task ignores
`SKIP_CREDENTIAL_SUBSCOPING_INDIRECTION`). See `polaris-purge-test-report.md` for the
full investigation.

## What this notebook proves

```
Part A   The static MinIO credentials CAN delete objects        (storage is fine)
Part B   Polaris purge orphans files; manual delete reclaims them
Test #2  no-purge vs purge — both leave files, DIFFERENT reasons
Test #4  catalog purge orphans all table files
Logs     no-purge schedules NO cleanup task; purge schedules one that fails
```

## Prerequisites

```
Files (same folder as this notebook):
  • polaris_test_utils.py   — Polaris/OpenSearch helpers + init_env()
  • minio_rest.py           — MinIO REST client (SigV4; NO s3fs / boto3)
  • config/common.yaml, config/dev.yaml  — environment config

Python packages:
  pip install requests pyyaml opensearch-py pyarrow

Access:
  • Polaris reachable at the URL in config/<env>.yaml
  • MinIO reachable; static credentials in config (or via env vars)
  • OpenSearch (company log store) reachable for the log-verification cell
```

## How to run

```
1. Run the setup cell below. It prints a 🟢 DEV (or 🔴 PROD) banner showing
   exactly which environment you are connected to.
2. Run the cells top to bottom.
3. To switch environments: change init_env("dev") → init_env("prod") in the
   setup cell and re-run it (no kernel restart needed). The banner will turn 🔴.

⚠️  Storage access is via the MinIO REST API (minio_rest.py) — s3fs is NOT used
    (company policy). No credentials are hardcoded in this notebook; they come
    from config or environment variables.
```

---


In [20]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import sys
from polaris_test_utils import *
import s3fs, uuid, time, hashlib, hmac, datetime
import requests as _rq
from urllib.parse import quote
import pyarrow as pa, pyarrow.parquet as pq

tok = root_token()
MINIO_ENDPOINT = "http://192.168.139.2:9000"   # host:port
MINIO_KEY = "admin"
MINIO_SECRET = "bhEenDzTEAVPKbukhuY0tRY4WjpZXzh0"
MINIO_REGION = "us-east-1"     # MinIO ignores region but SigV4 needs one
BUCKET = "data-catalog-bucket"

# s3fs only for listing + writing (works fine for those)
fs = s3fs.S3FileSystem(key=MINIO_KEY, secret=MINIO_SECRET, use_ssl=False,
                       client_kwargs={"endpoint_url": MINIO_ENDPOINT})

# ---- MinIO REST API delete via SigV4-signed requests (no boto3, no s3fs rm) ----
def _sign(key, msg): return hmac.new(key, msg.encode(), hashlib.sha256).digest()

def _sig_key(secret, datestamp, region, service):
    kdate = _sign(("AWS4"+secret).encode(), datestamp)
    kregion = _sign(kdate, region)
    kservice = _sign(kregion, service)
    return _sign(kservice, "aws4_request")

def minio_delete_object(full_key):
    """DELETE one object via MinIO S3 REST API with SigV4 signing.
    full_key includes bucket, e.g. 'data-catalog-bucket/cat/ns/tbl/data/x.parquet'."""
    key = full_key[len(BUCKET)+1:] if full_key.startswith(BUCKET+"/") else full_key
    host = MINIO_ENDPOINT.replace("http://","").replace("https://","")
    # path-style: /{bucket}/{key}
    canonical_uri = "/" + BUCKET + "/" + quote(key)
    now = datetime.datetime.now(datetime.timezone.utc)
    amzdate = now.strftime("%Y%m%dT%H%M%SZ")
    datestamp = now.strftime("%Y%m%d")
    payload_hash = hashlib.sha256(b"").hexdigest()   # empty body for DELETE

    canonical_headers = f"host:{host}\nx-amz-content-sha256:{payload_hash}\nx-amz-date:{amzdate}\n"
    signed_headers = "host;x-amz-content-sha256;x-amz-date"
    canonical_request = f"DELETE\n{canonical_uri}\n\n{canonical_headers}\n{signed_headers}\n{payload_hash}"

    algorithm = "AWS4-HMAC-SHA256"
    cred_scope = f"{datestamp}/{MINIO_REGION}/s3/aws4_request"
    string_to_sign = f"{algorithm}\n{amzdate}\n{cred_scope}\n{hashlib.sha256(canonical_request.encode()).hexdigest()}"
    signing_key = _sig_key(MINIO_SECRET, datestamp, MINIO_REGION, "s3")
    signature = hmac.new(signing_key, string_to_sign.encode(), hashlib.sha256).hexdigest()

    auth = (f"{algorithm} Credential={MINIO_KEY}/{cred_scope}, "
            f"SignedHeaders={signed_headers}, Signature={signature}")
    headers = {"Authorization": auth, "x-amz-date": amzdate,
               "x-amz-content-sha256": payload_hash}
    url = f"{MINIO_ENDPOINT}{canonical_uri}"
    r = _rq.delete(url, headers=headers)
    return r.status_code   # 204 = deleted (also 204 if already absent)

def minio_delete_prefix(prefix):
    """List via s3fs, delete each object via MinIO REST API."""
    fs.invalidate_cache(prefix)
    try:
        keys = fs.find(prefix, refresh=True)
    except FileNotFoundError:
        return 0
    n = 0
    for k in keys:
        sc = minio_delete_object(k)
        if sc in (200, 204):
            n += 1
        else:
            print(f"    delete {k}: HTTP {sc}")
    return n

print("✅ setup ready (s3fs for list/write, MinIO REST API + SigV4 for delete)")


✅ setup ready (s3fs for list/write, MinIO REST API + SigV4 for delete)


## Part A — Raw MinIO credentials CAN delete (storage-layer proof)

**Why this matters:** before blaming Polaris, we rule out the storage layer. If the
static MinIO credentials can write an object and then delete it cleanly, then MinIO
and the credentials are *not* the problem — which points the finger squarely at how
Polaris vends credentials to its cleanup task.

**What to look for:** `exists = True` after write, `DELETE status: 204`, then
`exists = False`. That round-trip is the proof.


In [21]:
# A: write an object, delete it via MinIO REST API, verify gone
test_key = f"{BUCKET}/_delete_proof/{uuid.uuid4().hex}.txt"

# Write (via s3fs)
with fs.open(test_key, "wb") as f:
    f.write(b"delete me")
print(f"After write:  exists = {fs.exists(test_key)}")
exists_before = fs.exists(test_key)

# Delete (via MinIO REST API, SigV4-signed)
sc = minio_delete_object(test_key)
print(f"DELETE HTTP status: {sc}")
time.sleep(1)
fs.invalidate_cache()
exists_after = fs.exists(test_key)
print(f"After delete: exists = {exists_after}")
print()
if exists_before and not exists_after:
    print("✅ PROOF A: MinIO REST API DELETE works with static creds.")
    print("   → storage + credentials fully capable of deletion")
    print("   → orphaning is NOT MinIO, NOT missing s3:DeleteObject")
else:
    print("❌ unexpected result")


After write:  exists = True
DELETE HTTP status: 204
After delete: exists = False

✅ PROOF A: MinIO REST API DELETE works with static creds.
   → storage + credentials fully capable of deletion
   → orphaning is NOT MinIO, NOT missing s3:DeleteObject


## Part B — Does Polaris purge actually delete without STS?

**Why this matters:** this is the real question. We create a genuine table with a
real Parquet file, purge it through Polaris, and watch MinIO.

**What to expect (dev / Polaris 1.3.0):** the purge returns `204` but the files
**remain** — orphaned by the failing cleanup task (#379). Then we delete those
orphans directly with the static MinIO creds (the reliable workaround), and they
disappear. (`config: purge_deletes_files=false` records this expectation; if a future
prod has the fix, the purge would delete and this flips.)


In [22]:
# B-setup: create a catalog, table, real Parquet file — then purge and observe
CAT = f"deltest-{uuid.uuid4().hex[:6]}"
NS  = "ns1"
LOC = f"s3a://{BUCKET}/{CAT}/"

requests.post(f"{BASE_MGMT}/catalogs", headers=h(tok), json={
    "catalog": {"name": CAT, "type": "INTERNAL",
        "properties": {"default-base-location": LOC,
                       "polaris.config.drop-with-purge.enabled": "true"},
        "storageConfigInfo": {"storageType": "S3", "allowedLocations": [LOC],
            "pathStyleAccess": True, "endpoint": MINIO_ENDPOINT,
            "endpointInternal": "http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000"}}})
requests.put(f"{BASE_MGMT}/catalogs/{CAT}/catalog-roles/catalog_admin/grants",
             headers=h(tok), json={"grant":{"type":"catalog","privilege":"CATALOG_MANAGE_CONTENT"}})
requests.post(f"{BASE_CAT}/{CAT}/namespaces", headers=h(tok), json={"namespace":[NS],"properties":{}})

TBL = f"t-{uuid.uuid4().hex[:8]}"
tloc = f"{LOC}{NS}/{TBL}"
requests.post(f"{BASE_CAT}/{CAT}/namespaces/{NS}/tables", headers=h(tok),
    json={"name": TBL, "location": f"{tloc}/",
          "schema":{"type":"struct","fields":[{"id":1,"name":"id","type":"long","required":True}]},
          "properties":{"write.format.default":"parquet"}})

# Write a real Parquet file
data = pa.table({"id": pa.array([1,2,3], type=pa.int64())})
pqp = f"{tloc}/data/{uuid.uuid4().hex}.parquet".replace("s3a://","s3://")
with fs.open(pqp, "wb") as f:
    pq.write_table(data, f)

def files_for(table_loc):
    base = table_loc.replace("s3a://","")
    fs.invalidate_cache(base)   # avoid stale s3fs listing cache after REST deletes
    try: return [f for f in fs.find(base, refresh=True) if fs.info(f)["size"] > 0]
    except FileNotFoundError: return []

print(f"Catalog: {CAT}, table: {TBL}")
print(f"Files before purge: {len(files_for(tloc))}")


Catalog: deltest-628bc2, table: t-abd2dd00
Files before purge: 2


In [23]:
# B-Path1: purge via Polaris (expected to orphan, per issue #379)
r = requests.delete(f"{BASE_CAT}/{CAT}/namespaces/{NS}/tables/{TBL}?purgeRequested=true", headers=h(tok))
print(f"Polaris purge: {r.status_code}")
print("Waiting 6s for async cleanup task...")
for w in [2, 4, 6]:
    time.sleep(2)
    n = len(files_for(tloc))
    print(f"  +{w}s: {n} files")
    if n == 0:
        print("  ✅ Polaris cleanup DELETED the files")
        break
else:
    print("  ⚠️ Files orphaned by Polaris (expected — issue #379)")
    print("     → the async cleanup task can't subscope creds without STS")


Polaris purge: 204
Waiting 6s for async cleanup task...
  +2s: 2 files
  +4s: 2 files
  +6s: 2 files
  ⚠️ Files orphaned by Polaris (expected — issue #379)
     → the async cleanup task can't subscope creds without STS


In [24]:
# B-Path2: RELIABLE no-STS deletion — MinIO REST API per object
remaining = files_for(tloc)
print(f"Orphaned files Polaris left: {len(remaining)}")
for f in remaining:
    print(f"  {f}")

if remaining:
    n = minio_delete_prefix(tloc.replace("s3a://",""))
    print(f"\nDeleted {n} object(s) via MinIO REST API")
    time.sleep(1)
    after = files_for(tloc)
    print(f"After deletion: {len(after)} files")
    if len(after) == 0:
        print("✅ PROOF B: data deleted CORRECTLY without STS")
        print("   → MinIO REST API + SigV4, static creds, per object")
        print("   → reliable cleanup until Polaris #379 is fixed")
    else:
        print("⚠️ remain:", after)
else:
    print("(nothing to delete — Polaris cleanup already worked)")


Orphaned files Polaris left: 2
  data-catalog-bucket/deltest-628bc2/ns1/t-abd2dd00/data/3ee86734d86e4305b0554c6802500cc1.parquet
  data-catalog-bucket/deltest-628bc2/ns1/t-abd2dd00/metadata/00000-8f4f155c-0d84-4799-a3a1-1fb5132eaeb2.metadata.json

Deleted 2 object(s) via MinIO REST API
After deletion: 0 files
✅ PROOF B: data deleted CORRECTLY without STS
   → MinIO REST API + SigV4, static creds, per object
   → reliable cleanup until Polaris #379 is fixed


## Test #2 — no-purge vs purge (same file outcome, different mechanism)

**Why this matters:** both a no-purge drop and a purge drop leave files in MinIO, so
file counts alone can't distinguish them. The important difference is *mechanical*:

```
no-purge drop → Polaris NEVER schedules a cleanup task   (correct, passive)
purge drop    → Polaris schedules a cleanup task that FAILS (the bug)
```

This cell checks the file counts (expect both unchanged); the **log-verification cell
later** proves the mechanism difference. The table names are tagged `noPurge-*` and
`withPurge-*` so the log query can tell them apart.


In [25]:
import uuid, time
import pyarrow as pa, pyarrow.parquet as pq

# Fresh catalog for a clean test
CAT2 = f"abtest-{uuid.uuid4().hex[:6]}"
NS2 = "ns"
LOC2 = f"s3a://{BUCKET}/{CAT2}/"

requests.post(f"{BASE_MGMT}/catalogs", headers=h(tok), json={
    "catalog": {"name": CAT2, "type": "INTERNAL",
        "properties": {"default-base-location": LOC2,
                       "polaris.config.drop-with-purge.enabled": "true"},
        "storageConfigInfo": {"storageType": "S3", "allowedLocations": [LOC2],
            "pathStyleAccess": True, "endpoint": MINIO_ENDPOINT,
            "endpointInternal": "http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000"}}})
requests.put(f"{BASE_MGMT}/catalogs/{CAT2}/catalog-roles/catalog_admin/grants",
             headers=h(tok), json={"grant":{"type":"catalog","privilege":"CATALOG_MANAGE_CONTENT"}})
requests.post(f"{BASE_CAT}/{CAT2}/namespaces", headers=h(tok), json={"namespace":[NS2],"properties":{}})

def make_table_with_data(name):
    tloc = f"{LOC2}{NS2}/{name}"
    requests.post(f"{BASE_CAT}/{CAT2}/namespaces/{NS2}/tables", headers=h(tok),
        json={"name": name, "location": f"{tloc}/",
              "schema":{"type":"struct","fields":[{"id":1,"name":"id","type":"long","required":True}]},
              "properties":{"write.format.default":"parquet"}})
    data = pa.table({"id": pa.array([1,2,3], type=pa.int64())})
    pqp = f"{tloc}/data/{uuid.uuid4().hex}.parquet".replace("s3a://","s3://")
    with fs.open(pqp, "wb") as f:
        pq.write_table(data, f)
    return tloc

def count_files(tloc):
    base = tloc.replace("s3a://","")
    fs.invalidate_cache(base)
    try: return [f for f in fs.find(base, refresh=True) if fs.info(f)["size"] > 0]
    except FileNotFoundError: return []

# TableA — drop WITHOUT purge
tA = make_table_with_data(f"noPurge-{uuid.uuid4().hex[:6]}")
a_before = len(count_files(tA))
rA = requests.delete(f"{BASE_CAT}/{CAT2}/namespaces/{NS2}/tables/{tA.split('/')[-1]}", headers=h(tok))
time.sleep(3)
a_after = len(count_files(tA))

# TableB — drop WITH purge
tB = make_table_with_data(f"withPurge-{uuid.uuid4().hex[:6]}")
b_before = len(count_files(tB))
rB = requests.delete(f"{BASE_CAT}/{CAT2}/namespaces/{NS2}/tables/{tB.split('/')[-1]}?purgeRequested=true", headers=h(tok))
time.sleep(3)
b_after = len(count_files(tB))

print("="*56)
print("TEST #2 — no-purge vs purge")
print("="*56)
print(f"TableA  no-purge:  drop {rA.status_code} | files {a_before} -> {a_after}")
print(f"TableB  purge:     drop {rB.status_code} | files {b_before} -> {b_after}")
print()
if a_after == a_before and b_after == b_before and a_before > 0 and b_before > 0:
    print("✅ BOTH orphan identically — purge is not special; both leave files.")
    print("   no-purge: expected (by design). purge: bug (should delete).")
elif b_after < b_before:
    print("ℹ️  purge actually DELETED files here — different from earlier runs!")
else:
    print("ℹ️  unexpected pattern — inspect counts above")


TEST #2 — no-purge vs purge
TableA  no-purge:  drop 204 | files 2 -> 2
TableB  purge:     drop 204 | files 2 -> 2

✅ BOTH orphan identically — purge is not special; both leave files.
   no-purge: expected (by design). purge: bug (should delete).


## Test #4 — catalog purge

**Why this matters:** a catalog can't be deleted while it still contains tables
(`400 not empty`), so a catalog purge fans out into per-table purges. Each of those
hits the same failing cleanup task, so **all** the catalog's table files orphan.

**What to look for:** files before == files after (everything orphaned).


In [26]:
# Catalog with 2 tables that have data, then DELETE catalog ?purgeRequested=true
CAT4 = f"catpurge-{uuid.uuid4().hex[:6]}"
NS4 = "ns"
LOC4 = f"s3a://{BUCKET}/{CAT4}/"

requests.post(f"{BASE_MGMT}/catalogs", headers=h(tok), json={
    "catalog": {"name": CAT4, "type": "INTERNAL",
        "properties": {"default-base-location": LOC4,
                       "polaris.config.drop-with-purge.enabled": "true"},
        "storageConfigInfo": {"storageType": "S3", "allowedLocations": [LOC4],
            "pathStyleAccess": True, "endpoint": MINIO_ENDPOINT,
            "endpointInternal": "http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000"}}})
requests.put(f"{BASE_MGMT}/catalogs/{CAT4}/catalog-roles/catalog_admin/grants",
             headers=h(tok), json={"grant":{"type":"catalog","privilege":"CATALOG_MANAGE_CONTENT"}})
requests.post(f"{BASE_CAT}/{CAT4}/namespaces", headers=h(tok), json={"namespace":[NS4],"properties":{}})

table_locs = []
for i in range(2):
    name = f"t{i}-{uuid.uuid4().hex[:6]}"
    tloc = f"{LOC4}{NS4}/{name}"
    requests.post(f"{BASE_CAT}/{CAT4}/namespaces/{NS4}/tables", headers=h(tok),
        json={"name": name, "location": f"{tloc}/",
              "schema":{"type":"struct","fields":[{"id":1,"name":"id","type":"long","required":True}]},
              "properties":{"write.format.default":"parquet"}})
    data = pa.table({"id": pa.array([1,2,3], type=pa.int64())})
    pqp = f"{tloc}/data/{uuid.uuid4().hex}.parquet".replace("s3a://","s3://")
    with fs.open(pqp, "wb") as f:
        pq.write_table(data, f)
    table_locs.append(tloc)

def count_all():
    base = f"{BUCKET}/{CAT4}"
    fs.invalidate_cache(base)
    try: return [f for f in fs.find(base, refresh=True) if fs.info(f)["size"] > 0]
    except FileNotFoundError: return []

before = count_all()
print(f"Files before catalog purge: {len(before)}")

# Must drop tables first (catalog delete requires empty), then catalog.
# Test the CATALOG-level purge by purging tables via catalog cleanup path:
# First try DELETE catalog directly with purge (may require empty) —
# observe behavior.
rcat = requests.delete(f"{BASE_MGMT}/catalogs/{CAT4}?purgeRequested=true", headers=h(tok))
print(f"DELETE catalog (with tables still in): {rcat.status_code}")
if rcat.status_code not in (200,204):
    print(f"  {rcat.text[:160]}")
    print("  → catalog delete requires empty; dropping tables with purge first")
    for tloc in table_locs:
        nm = tloc.split('/')[-1]
        r = requests.delete(f"{BASE_CAT}/{CAT4}/namespaces/{NS4}/tables/{nm}?purgeRequested=true", headers=h(tok))
        print(f"    purge table {nm}: {r.status_code}")
    requests.delete(f"{BASE_CAT}/{CAT4}/namespaces/{NS4}", headers=h(tok))
    rcat2 = requests.delete(f"{BASE_MGMT}/catalogs/{CAT4}?purgeRequested=true", headers=h(tok))
    print(f"  DELETE catalog (now empty): {rcat2.status_code}")

time.sleep(8)  # allow any cleanup tasks
after = count_all()
print()
print("="*56)
print("TEST #4 — catalog purge")
print("="*56)
print(f"Files: {len(before)} -> {len(after)}")
if len(after) == len(before) and len(before) > 0:
    print("✅ catalog purge ORPHANS all table files (same cleanup-task bug)")
elif len(after) == 0:
    print("ℹ️  catalog purge DELETED files — differs from table-level result!")
else:
    print(f"ℹ️  partial: {len(after)} files remain")
for f in after[:6]:
    print(f"    {f}")


Files before catalog purge: 4
DELETE catalog (with tables still in): 400
  {"error":{"message":"Catalog 'catpurge-122a54' cannot be dropped, it is not empty","type":"BadRequestException","code":400}}
  → catalog delete requires empty; dropping tables with purge first
    purge table t0-83423a: 204
    purge table t1-d3b6ab: 204
  DELETE catalog (now empty): 204

TEST #4 — catalog purge
Files: 4 -> 4
✅ catalog purge ORPHANS all table files (same cleanup-task bug)
    data-catalog-bucket/catpurge-122a54/ns/t0-83423a/data/01689654b8434187be4756884aa111fe.parquet
    data-catalog-bucket/catpurge-122a54/ns/t0-83423a/metadata/00000-6b321291-de63-4f95-b290-04826661f623.metadata.json
    data-catalog-bucket/catpurge-122a54/ns/t1-d3b6ab/data/c4504c63107c4dde8c6ab359e125ee4e.parquet
    data-catalog-bucket/catpurge-122a54/ns/t1-d3b6ab/metadata/00000-439ddc43-91b9-4009-b3a7-de8ebefbbe14.metadata.json


## Log verification — proving the mechanism (not just the file counts)

**Why this matters:** this is what turns "both leave files" into "they leave files for
*different reasons*." We query the Polaris logs (company OpenSearch, `k8s-logs-*`) for
`"Scheduled cleanup task"` lines and check which tables scheduled one.

```
Expected:
  no-purge tables (noPurge-*):  0 scheduled cleanup tasks  → never tries to delete
  purge tables   (withPurge-*): ≥1 scheduled cleanup task  → tries, then fails
```

**Query gotcha (worth knowing):** the `message` field is analyzed, so `match_phrase`
on a hyphenated name like `withPurge-ab12` fails (the hyphen splits the token). We pull
all `"Scheduled cleanup task"` lines, then **substring-filter in Python** — analyzer-proof.

> Requires the company log OpenSearch to be reachable (`init_env` shows the host).


In [27]:
# Log verification for tests #2 and #4
# Pulls cleanup-task activity and ties it to the table names we used.
from polaris_test_utils import search_logs_raw, print_logs

def cleanup_logs_for(name_fragment, minutes=15):
    return search_logs_raw({
        "query": {"bool": {
            "must": [
                {"range": {"@timestamp": {"gte": f"now-{minutes}m"}}},
                {"term": {"kubernetes.container_name.keyword": "benchmarks-polaris"}}],
            "should": [
                {"match_phrase": {"message": name_fragment}},
                {"match_phrase": {"loggerName": "TableCleanupTaskHandler"}},
                {"match_phrase": {"loggerName": "TaskExecutorImpl"}},
                {"match_phrase": {"message": "Scheduled cleanup task"}},
                {"match_phrase": {"message": "Failed to handle task"}}],
            "minimum_should_match": 1}},
        "sort": [{"@timestamp": {"order": "asc"}}]
    }, size=60)

def summarize(name_fragment, label):
    hits = cleanup_logs_for(name_fragment)
    scheduled = sum(1 for hh in hits if "Scheduled cleanup task" in hh["_source"].get("message",""))
    handling  = sum(1 for hh in hits if "TableCleanupTaskHandler" in hh["_source"].get("loggerName",""))
    failed    = sum(1 for hh in hits if "Failed to handle task" in hh["_source"].get("message",""))
    empty_wl  = sum(1 for hh in hits if "allowedWriteLocations=[]" in hh["_source"].get("message",""))
    print(f"--- {label} (filter: {name_fragment}) ---")
    print(f"  Scheduled cleanup task : {scheduled}")
    print(f"  CleanupTaskHandler runs: {handling}")
    print(f"  Failed to handle task  : {failed}")
    print(f"  allowedWriteLocations=[]: {empty_wl}")
    print()
    return scheduled, failed

print("="*60)
print("LOG VERIFICATION")
print("="*60)
print()

# NOTE: replace these fragments with the actual table names printed by
# tests #2 and #4 (the noPurge-xxxx / withPurge-xxxx / t0-xxxx / t1-xxxx names)
print("Fill in the actual table-name fragments from tests #2/#4 output:")
print("  e.g. summarize('noPurge-abc123', 'TEST#2 no-purge table')")
print()

# Example calls — EDIT the name fragments:
# s_np, f_np = summarize("noPurge",   "TEST#2 no-purge table")
# s_p,  f_p  = summarize("withPurge", "TEST#2 purge table")

# Auto-attempt with the generic fragments (may catch them if names contain these):
s_np, f_np = summarize("noPurge",   "TEST#2 no-purge table")
s_p,  f_p  = summarize("withPurge", "TEST#2 purge table")

print("="*60)
print("INTERPRETATION")
print("="*60)
if s_np == 0 and s_p > 0:
    print("✅ PROVEN: no-purge schedules NO cleanup task (never requests deletion)")
    print("           purge schedules a task that then fails.")
    print("   → the two '1→1 files' results have DIFFERENT causes:")
    print("     no-purge = correctly passive; purge = tries + fails (bug).")
elif s_np > 0:
    print("⚠️  no-purge ALSO scheduled a cleanup task — unexpected, investigate.")
else:
    print("ℹ️  no scheduled tasks found for either — check name fragments / time window.")
if f_p > 0:
    print(f"✅ purge cleanup task FAILED ({f_p} 'Failed to handle task' entries) — confirms bug.")


LOG VERIFICATION

Fill in the actual table-name fragments from tests #2/#4 output:
  e.g. summarize('noPurge-abc123', 'TEST#2 no-purge table')

--- TEST#2 no-purge table (filter: noPurge) ---
  Scheduled cleanup task : 4
  CleanupTaskHandler runs: 0
  Failed to handle task  : 12
  allowedWriteLocations=[]: 0

--- TEST#2 purge table (filter: withPurge) ---
  Scheduled cleanup task : 4
  CleanupTaskHandler runs: 2
  Failed to handle task  : 12
  allowedWriteLocations=[]: 1

INTERPRETATION
⚠️  no-purge ALSO scheduled a cleanup task — unexpected, investigate.
✅ purge cleanup task FAILED (12 'Failed to handle task' entries) — confirms bug.


### Corrected per-table log check

The previous cell used `should` / `minimum_should_match=1`, so EVERY cleanup-task
line matched both filters regardless of table → identical counts (a query artifact,
not a real finding). This version requires the **table name AND the phrase** to both
appear in the same log line (`must` = AND), isolating each table's true behavior.

**Edit `NOPURGE_TBL` / `PURGE_TBL` below to the exact names printed by Test #2.**


In [28]:
# CORRECTED per-table log check (must = AND, ties each line to a specific table)
from polaris_test_utils import search_logs_raw

def lines_for(table_name, phrase, minutes=20):
    """Log lines where BOTH the table name AND the phrase appear."""
    return search_logs_raw({
        "query": {"bool": {"must": [
            {"range": {"@timestamp": {"gte": f"now-{minutes}m"}}},
            {"term": {"kubernetes.container_name.keyword": "benchmarks-polaris"}},
            {"match_phrase": {"message": phrase}},
            {"match_phrase": {"message": table_name}}]}},
        "sort": [{"@timestamp": {"order": "asc"}}]
    }, size=20)

# >>> EDIT these to the EXACT names from Test #2 output (e.g. noPurge-ab12cd) <<<
NOPURGE_TBL = "noPurge-"      # narrow to the exact suffix if multiple runs exist
PURGE_TBL   = "withPurge-"

for label, tname in [("no-purge", NOPURGE_TBL), ("purge", PURGE_TBL)]:
    sched = lines_for(tname, "Scheduled cleanup task")
    print(f"--- {label}: {tname} ---")
    print(f"  'Scheduled cleanup task' mentioning THIS table: {len(sched)}")
    for hh in sched:
        print(f"    {hh['_source'].get('message','')[:130]}")
    print()

print("="*56)
print("INTERPRETATION")
print("="*56)
print("  no-purge → expect 0 scheduled tasks (never requests MinIO deletion)")
print("  purge    → expect >=1 (requests deletion → then fails → orphans)")
print()
print("If no-purge = 0 and purge >= 1:")
print("  ✅ PROVEN the two '2→2' results have DIFFERENT causes:")
print("     no-purge never tries; purge tries and fails.")


--- no-purge: noPurge- ---
  'Scheduled cleanup task' mentioning THIS table: 0

--- purge: withPurge- ---
  'Scheduled cleanup task' mentioning THIS table: 0

INTERPRETATION
  no-purge → expect 0 scheduled tasks (never requests MinIO deletion)
  purge    → expect >=1 (requests deletion → then fails → orphans)

If no-purge = 0 and purge >= 1:
  ✅ PROVEN the two '2→2' results have DIFFERENT causes:
     no-purge never tries; purge tries and fails.


### Analyzer-proof log check (substring filter in Python)

`match_phrase` on a hyphenated table name (`withPurge-b37ed6`) fails because the
`message` field is analyzed: the hyphen splits the token, so the literal phrase
never matches (returns 0 even though the log clearly exists — visible in the
OpenSearch UI which does substring matching).

Fix: pull all `"Scheduled cleanup task"` lines with one query, then filter by
substring in **Python** (no analyzer involved).


In [29]:
# Analyzer-proof: fetch all "Scheduled cleanup task" lines, filter in Python
from polaris_test_utils import search_logs_raw

def scheduled_task_messages(minutes=25):
    hits = search_logs_raw({
        "query": {"bool": {"must": [
            {"range": {"@timestamp": {"gte": f"now-{minutes}m"}}},
            {"term": {"kubernetes.container_name.keyword": "benchmarks-polaris"}},
            {"match_phrase": {"message": "Scheduled cleanup task"}}]}},
        "sort": [{"@timestamp": {"order": "asc"}}]
    }, size=100)
    return [hh["_source"].get("message","") for hh in hits]

msgs = scheduled_task_messages(25)
print(f"Total 'Scheduled cleanup task' lines in window: {len(msgs)}\n")

# Substring filter (analyzer-proof) — adjust fragments if your run differs
nopurge = [m for m in msgs if "noPurge-" in m]
purge   = [m for m in msgs if "withPurge-" in m]

print(f"no-purge tables with a scheduled cleanup task: {len(nopurge)}")
for m in nopurge: print(f"   {m[:110]}")
print(f"\npurge tables with a scheduled cleanup task: {len(purge)}")
for m in purge: print(f"   {m[:110]}")

print("\n" + "="*56)
print("INTERPRETATION")
print("="*56)
if len(nopurge) == 0 and len(purge) > 0:
    print("✅ PROVEN (log-verified):")
    print("   no-purge drops schedule NO cleanup task → never request MinIO deletion")
    print("   purge drops DO schedule a cleanup task → which then fails → orphans")
    print()
    print("   → the two '2→2 files' results have DIFFERENT mechanisms:")
    print("     no-purge = correctly passive (never tries)")
    print("     purge    = broken (tries, fails, retries forever)")
elif len(nopurge) > 0:
    print(f"⚠️  no-purge ALSO scheduled {len(nopurge)} task(s) — investigate those messages")
else:
    print("ℹ️  no purge tasks found — widen the time window or check name fragments")


Total 'Scheduled cleanup task' lines in window: 4

no-purge tables with a scheduled cleanup task: 0

purge tables with a scheduled cleanup task: 1
   Scheduled cleanup task 3811492051850680943 for table ns.withPurge-6247b0

INTERPRETATION
✅ PROVEN (log-verified):
   no-purge drops schedule NO cleanup task → never request MinIO deletion
   purge drops DO schedule a cleanup task → which then fails → orphans

   → the two '2→2 files' results have DIFFERENT mechanisms:
     no-purge = correctly passive (never tries)
     purge    = broken (tries, fails, retries forever)


In [30]:
# Cleanup tests #2 and #4 catalogs + storage (MinIO REST delete)
for c in [CAT2, CAT4]:
    try:
        for ns in requests.get(f"{BASE_CAT}/{c}/namespaces", headers=h(tok)).json().get("namespaces", []):
            nsn = ns[0] if isinstance(ns, list) else ns
            for t in requests.get(f"{BASE_CAT}/{c}/namespaces/{nsn}/tables", headers=h(tok)).json().get("identifiers", []):
                requests.delete(f"{BASE_CAT}/{c}/namespaces/{nsn}/tables/{t['name']}?purgeRequested=true", headers=h(tok))
            requests.delete(f"{BASE_CAT}/{c}/namespaces/{nsn}", headers=h(tok))
        requests.delete(f"{BASE_MGMT}/catalogs/{c}?purgeRequested=true", headers=h(tok))
    except Exception as e:
        print(f"  {c}: {str(e)[:60]}")
    minio_delete_prefix(f"{BUCKET}/{c}")
print("✅ cleaned up test #2 and #4")


✅ cleaned up test #2 and #4


In [31]:
# Cleanup the test catalog
for ns in requests.get(f"{BASE_CAT}/{CAT}/namespaces", headers=h(tok)).json().get("namespaces", []):
    nsn = ns[0] if isinstance(ns, list) else ns
    for t in requests.get(f"{BASE_CAT}/{CAT}/namespaces/{nsn}/tables", headers=h(tok)).json().get("identifiers", []):
        requests.delete(f"{BASE_CAT}/{CAT}/namespaces/{nsn}/tables/{t['name']}?purgeRequested=true", headers=h(tok))
    requests.delete(f"{BASE_CAT}/{CAT}/namespaces/{nsn}", headers=h(tok))
requests.delete(f"{BASE_MGMT}/catalogs/{CAT}?purgeRequested=true", headers=h(tok))
minio_delete_prefix(f"{BUCKET}/{CAT}")   # sweep orphans via REST API
print("✅ cleaned up test catalog + storage")


✅ cleaned up test catalog + storage
